#### Benchmark subtract_z_motion_pixels
Test with T-series recorded with no calcium activity, only z motion.  
  
T-series data: `C57_O1M1/09182023/TSeries-09182023-1503-003`  
Z-series: `ZSeries-09182023-1503-003`  

In [ ]:
from pathlib import Path
import json
import os
import numpy as np
import mesmerize_core as mc

import sys
sys.path.append("..")  # Add scripts directory to python modules path.
import pipeline_mcorr_cnmf as pmc
import compute_zcorr as cz

path_file = '/home/wanglab/code/imaging_analysis/Analysis_2P/test/Paths_zmotion_lin.json'

with open(path_file) as f:
    paths = json.load(f)
    
params_files = paths['params_files']
with open(params_files[0]) as f:
    parameters = json.load(f)

zstack_path = paths['zstack_paths']
zstack_path = Path(zstack_path[0])
z_params_files = paths['z_params_files']
with open(z_params_files[0]) as f:
    z_parameters = json.load(f)
    
z_shifted_file = z_parameters['zstack_shift']['file_name']

# Get the motion correction parameters
parameters_mcorr = parameters['params_mcorr']

In [ ]:
# Run the motion correction 
data_paths = paths['data_paths']
export_paths = paths['export_paths']
regex_pattern='*_Ch2_*.ome.tif'

export_path= Path(export_paths[0])
# If export path does not exist, create it
if not export_path.exists():
    os.makedirs(export_path)
# Set the parent raw data path
mc.set_parent_raw_data_path(export_path) 

# Run the x/y motion correction
batch_path, index, mcorr_movie_path = pmc.run_mcorr(data_paths, export_path, parameters_mcorr, regex_pattern)

In [ ]:
# Shift zstack
if not os.path.exists(zstack_path / z_shifted_file):
    shift_zstack_path=cz.shift_zstack(z_parameters['zstack_shift'], zstack_path, z_shifted_file)
else:
    shift_zstack_path=zstack_path / z_shifted_file

# Compute z-correlation
mesmerize_path = mcorr_movie_path.parents[1] # get mesmerize directory where the zcorr file will be saved
if not os.path.exists(mesmerize_path / "z_correlation.npy"):
    z_correlation=cz.compute_zpos(zstack_path / z_shifted_file, mcorr_movie_path)
    # np.save(mesmerize_path / "z_correlation.npy", z_correlation)   
else:
    z_correlation=np.load(mesmerize_path / "z_correlation.npy")

In [ ]:
from caiman.mmapping import load_memmap
from tqdm import tqdm
from scipy.stats import mode
from scipy.ndimage import gaussian_filter
from skimage.transform import warp
from skimage.registration import phase_cross_correlation
from PIL import Image
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
import multiprocessing

# Load the movie
movie_16bit, dims, T = load_memmap(mcorr_movie_path)
pixels = np.reshape(movie_16bit.T, [T] + list(dims), order='F')
Nframe, Nx, Ny = pixels.shape

# Load the FOV file
mcorr_batch_dir = mcorr_movie_path.parent
fov_file_path = Path.joinpath(mcorr_batch_dir, f"{mcorr_batch_dir.stem}_mean_projection.npy")
fov_image = np.load(fov_file_path)

# Load the z-stack
info_zstack = Image.open(shift_zstack_path)
Nz = info_zstack.n_frames
Zstack = np.zeros((Ny, Nx, Nz), dtype=np.float32)
for iz in range(Nz):
    info_zstack.seek(iz)
    Zstack[:, :, iz] = np.array(info_zstack)
info_zstack.close()        
# TODO: check why on Windows F_anat bit depth is 8 and not 16

# Flip Zstack up-down to match the orientation of the movie
Zstack = np.flip(Zstack, axis=0)

In [ ]:
# Plot the first image of the movie, the FOV image, and the top slice of the z-stack side by side
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 3, figsize=(15, 5))
ax[0].imshow(pixels[0], cmap='gray')
ax[0].set_title("Movie")
ax[1].imshow(fov_image, cmap='gray')
ax[1].set_title("FOV")
ax[2].imshow(Zstack[:, :, -1], cmap='gray')
ax[2].set_title("Z-stack")
plt.show()


In [ ]:

# Translate zstack frames to minimize shift with mean of registered tseries frames
zpos = z_correlation
zpos_0 = mode(zpos).mode
Zstack_0 = Zstack[:, :, zpos_0]

# Perform image registration
shift, _ , _ = phase_cross_correlation(Zstack_0, fov_image, upsample_factor=10)

# Create the transformation matrix (for translation only)
tform = np.eye(3)
tform[0, 2] = -shift[1]
tform[1, 2] = -shift[0]

# Translate z-stack frames to minimize shift with mean of registered tseries frames (FOV)
for iz in range(Nz):
    Zstack[:, :, iz] = warp(Zstack[:, :, iz], tform)

In [ ]:
# Make a movie of the z-stack frames that follows zcorr (movement in depth)
F_anat = Zstack[:, :, zpos]

In [ ]:
#  Plot the first image of F_anat with smoothing_factor 1 to 5 
fig, ax = plt.subplots(2, 3, figsize=(20, 10))
for i in range(6):
    row = i // 3  # Integer division gives the row index
    col = i % 3   # Modulo operation gives the column index
    ax[row, col].imshow(gaussian_filter(F_anat[:, :, 0], sigma=i), cmap='gray')
    ax[row, col].set_title(f"Smoothing factor: {i}")
plt.show()

In [ ]:
# Apply Gaussian smoothing to the functional fluorescence frames before reshaping
# Run the following cells with different values of smoothing_factor to see the effect of smoothing, starting from 0
smoothing_factor = 3

if smoothing_factor > 0:
    pixels_smoothed = np.zeros_like(pixels)
    for i in range(Nframe):
        pixels_smoothed[i] = gaussian_filter(pixels[i], sigma=smoothing_factor)
else:
    pixels_smoothed = pixels
    
# Reshape the "functional" fluorescence movie
F_func = pixels_smoothed.reshape(Nframe, Nx * Ny).T
    
# Get the average image of the functional fluorescence movie
F_func_mean = np.mean(F_func, axis=1)


In [ ]:
# #  Plot the first image of pixels and pixels_smoothed
# fig, ax = plt.subplots(1, 2, figsize=(10, 5))
# ax[0].imshow(pixels[0], cmap='gray')
# ax[0].set_title("Original movie")
# ax[1].imshow(pixels_smoothed[0], cmap='gray')
# ax[1].set_title("Smoothed movie")
# plt.show()

In [ ]:
# Apply Gaussian smoothing to the anatomical fluorescence frames before reshaping
F_anat = Zstack[:, :, zpos]
if smoothing_factor > 0:
    F_anat_smoothed = np.zeros_like(F_anat)
    for i in range(F_anat.shape[2]):
        F_anat_smoothed[:, :, i] = gaussian_filter(F_anat[:, :, i], sigma=smoothing_factor)
else:
    F_anat_smoothed = F_anat

In [ ]:
# Reshape the "anatomical" fluorescence movie
F_anat = F_anat_smoothed.reshape(Nx * Ny, Nframe)

# Get the average image of the z-stack movie
F_anat_mean = np.mean(F_anat, axis=1)

In [ ]:
from sklearn.linear_model import HuberRegressor

def fit_huber_regressor(i_pixel, F_anat, F_anat_mean, F_func, F_func_mean):
    """
    Fit a Huber regressor to find the scaling factor
    """
    huber = HuberRegressor(fit_intercept=False)
    try:
        huber.fit(F_anat[i_pixel, :].reshape(-1, 1) - F_anat_mean[i_pixel], F_func[i_pixel, :] - F_func_mean[i_pixel])
        b = huber.coef_[0]
    except ValueError as e:
        if "HuberRegressor convergence failed" in str(e):
            b = 0
            print(f"Convergence warning for pixel {i_pixel}. Setting b[i_pixel] to 0.")
        else:
            raise e
    return b

In [ ]:
n_jobs=-1
z_motion_scaling_factors = Parallel(n_jobs=n_jobs)(delayed(fit_huber_regressor)(i_pixel, F_anat, F_anat_mean, F_func, F_func_mean) for i_pixel in tqdm(range(Nx * Ny), desc='Fitting regressors to find z motion scaling factors'))
# Kill all LokyProcess workers (Windows only)
if os.name == 'nt':
    for p in multiprocessing.active_children():
        if 'LokyProcess' in p.name:
            p.terminate()
# Change the list to a numpy array and cast to float32
z_motion_scaling_factors = np.array(z_motion_scaling_factors)
z_motion_scaling_factors = z_motion_scaling_factors.astype(np.float32)

In [ ]:
z_motion_scaling_factors.shape

In [ ]:
# Rescale and subtract the movement-induced changes from the original F matrix

# First get back the non-smoothed F_anat and F_func
F_anat = Zstack[:, :, zpos].reshape(Nx * Ny, Nframe)
F_anat_mean = np.mean(F_anat, axis=1)
F_func = pixels.reshape(Nframe, Nx * Ny).T

# Rescale and subtract the movement-induced changes from the original F matrix
F_anat_rescaled = z_motion_scaling_factors[:, np.newaxis] * (F_anat - F_anat_mean[:, np.newaxis])
Fcorrected = F_func - F_anat_rescaled
Fcorrected = np.clip(Fcorrected, 0, 2**16-1)

# If z is >+5um or <-5um away from mode, then replace F by NaN and interpolate linearly from non missing values
missing_F = np.abs(zpos - zpos_0) > 5
if np.any(missing_F):
    Fcorrected[:, missing_F] = np.nan
    Fcorrected = pd.DataFrame(Fcorrected).interpolate(method='linear', axis=1, limit_direction='both').values
    # TODO: check that it only interpolates NaNs

In [ ]:
# Get range (min / max) and type of F_func and F_anat
F_func_range = (F_func.min(), F_func.max())
F_anat_range = (F_anat.min(), F_anat.max())

F_func_dtype = F_func.dtype
F_anat_dtype = F_anat.dtype

print(f"F_func range: {F_func_range}, dtype: {F_func_dtype}")
print(f"F_anat range: {F_anat_range}, dtype: {F_anat_dtype}")

In [ ]:
# Reshape Fcorrected to the original shape
F_func_reshaped = F_func.T.reshape(Nframe, Nx, Ny)
F_func_reshaped = np.clip(F_func_reshaped, 0, 2**16-1)
F_anat_rescaled_reshaped = F_anat_rescaled.T.reshape(Nframe, Nx, Ny)
min_Fars = np.min(F_anat_rescaled_reshaped)
max_Fars = np.max(F_anat_rescaled_reshaped)
F_anat_rescaled_reshaped = (F_anat_rescaled_reshaped - min_Fars) / (max_Fars - min_Fars) * (2**16-1)
Fcorrected_reshaped = Fcorrected.T.reshape(Nframe, Nx, Ny)
Fcorrected_reshaped = np.clip(Fcorrected_reshaped, 0, 2**16-1)
F_anat_reshaped = F_anat.T.reshape(Nframe, Nx, Ny)

In [ ]:
# Save an excerpt of the original F matrix to a video file (if not already saved)
# Take only the first 80 frames
F_func_reshaped_excerpt = F_func_reshaped[:80]
# Assign data type to uint16
F_func_reshaped_excerpt = F_func_reshaped_excerpt.astype(np.uint16)
# Save the array to a video file
# Convert each frame to an Image object
images = [Image.fromarray(frame) for frame in F_func_reshaped_excerpt]
# Save the first image and append the rest
images[0].save(export_path / 'F_notcorrected_excerpt.tif', save_all=True, append_images=images[1:], compression='tiff_lzw')

In [ ]:
# Save an excerpt of the corrected F matrix to a video file
# Take only the first 80 frames
Fcorrected_reshaped_excerpt = Fcorrected_reshaped[:80]
# Assign data type to uint16
Fcorrected_reshaped_excerpt = Fcorrected_reshaped_excerpt.astype(np.uint16)
# Save the array to a video file
# Convert each frame to an Image object
images = [Image.fromarray(frame) for frame in Fcorrected_reshaped_excerpt]
# Save the first image and append the rest
images[0].save(mcorr_movie_path.parent.parent / f"Fcorrected_smoothing_{smoothing_factor}.tif", save_all=True, append_images=images[1:], compression='tiff_lzw')

In [ ]:
# Once the different zcorr movies are saved, load them and concatenate them to create a video
export_path = mcorr_movie_path.parent.parent

# Load the excerpt of the original F matrix
F_func_excerpt_path = export_path / 'F_func_excerpt.tif'
F_func_excerpt = Image.open(F_func_excerpt_path)
F_func_excerpt_frames = []
try:
    while True:
        F_func_excerpt_frames.append(F_func_excerpt.copy())
        F_func_excerpt.seek(F_func_excerpt.tell() + 1)
except EOFError:
    pass

# Initialize a dictionary to store the excerpts
Fcorrected_excerpts = {}

# Load the excerpt of the corrected F matrices
for smoothing_factor in range(4):
    Fcorrected_excerpt_path = export_path / f"Fcorrected_smoothing_{smoothing_factor}.tif"
    Fcorrected_excerpt = Image.open(Fcorrected_excerpt_path)
    Fcorrected_excerpt_frames = []
    try:
        while True:
            Fcorrected_excerpt_frames.append(Fcorrected_excerpt.copy())
            Fcorrected_excerpt.seek(Fcorrected_excerpt.tell() + 1)
    except EOFError:
        pass
    # Store the frames in the dictionary
    Fcorrected_excerpts[f"Fcorrected_smoothing_{smoothing_factor}"] = Fcorrected_excerpt_frames
    
# Concatenate the excerpts horizontally
F_concat = np.concatenate([np.array(F_func_excerpt_frames), np.array(Fcorrected_excerpts['Fcorrected_smoothing_0']), np.array(Fcorrected_excerpts['Fcorrected_smoothing_1']), np.array(Fcorrected_excerpts['Fcorrected_smoothing_2']), np.array(Fcorrected_excerpts['Fcorrected_smoothing_3'])], axis=2)

# Save the concatenated video
F_concat_path = export_path / 'F_concatenated.tif'
# Convert each frame to an Image object
images = [Image.fromarray(frame) for frame in F_concat]
# Save the first image and append the rest
images[0].save(F_concat_path, save_all=True, append_images=images[1:])